In [31]:
import numpy as np
import os
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
osd_path = "\\Users\\User\\OneDrive\\Documents\\obsidian-git-sync\\Data\\"

today = date.today()
today

datetime.date(2023, 2, 9)

### Yesterday = Last business day

In [2]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(2)
yesterday = today - num_business_days
yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-02-09
yesterday: 2023-02-07


In [3]:
cols = 'name fm_date to_date fm_price to_price qty max_price min_price percent status'.split()

format_dict = {
    'fm_price':'{:.2f}','to_price':'{:.2f}','diff':'{:.2f}',
    'max_price':'{:.2f}','min_price':'{:.2f}',
    'volume':'{:,.2f}','beta':'{:,.2f}',
    'pct':'{:,.2f}%','percent':'{:,.2f}%',   
    'fm_date':'{:%Y-%m-%d}','to_date':'{:%Y-%m-%d}',
    'created_at':'{:%Y-%m-%d}','updated_at':'{:%Y-%m-%d}',
    
    'qty':'{:,}','available_qty':'{:,}',
    'cost':'{:.2f}','buy_target':'{:.2f}','sell_target':'{:.2f}',
}

### Begin of Tables in the process

In [4]:
sql = """
SELECT * 
FROM sales 
ORDER BY id DESC LIMIT 1
"""
tmp = pd.read_sql(sql, conlite)
tmp["fm_date"] = pd.to_datetime(tmp["fm_date"])
tmp["to_date"] = pd.to_datetime(tmp["to_date"])
tmp["created_at"] = pd.to_datetime(tmp["created_at"])
tmp["updated_at"] = pd.to_datetime(tmp["updated_at"])
tmp.style.format(format_dict)

,id,name,fm_date,to_date,days,fm_price,to_price,diff,pct,ttl_spread,avg_spread,max_price,min_price,qty,created_at,updated_at,latest_date_id
0,1216127,TMT,2023-02-07,2023-02-07,0,8.15,8.15,0.00,0.00%,0,0,8.20,8.10,"121,777",2023-02-07,2023-02-07,1


In [5]:
names = tmp["name"]
type(names)

pandas.core.series.Series

In [6]:
name = names.to_string(index=False)
type(name)

str

In [7]:
sql = """
SELECT * 
FROM stocks
WHERE name = '%s'
"""
sql = sql % name
print(sql)
tmp = pd.read_sql(sql, conlite)
tmp.style.format(format_dict)


SELECT * 
FROM stocks
WHERE name = 'TMT'



,id,name,max_price,min_price,status,buy_target,sell_target,volume,beta,cost,qty,buy_spread,sell_spread,available_qty,bl,sh,reason,market
0,645,TMT,15.00,9.70,S,7.40,8.70,13.30,1.01,9.90,"12,000",-16,15,"48,000",0,0,C9.0,SET


### End of Tables in the process

In [8]:
sql = """
SELECT DISTINCT a.name
FROM sales a 
WHERE to_date = '%s'
ORDER BY a.name
"""
sql = sql % yesterday
print(sql)
tmp = pd.read_sql(sql, conlite)
tmp


SELECT DISTINCT a.name
FROM sales a 
WHERE to_date = '2023-02-07'
ORDER BY a.name



,name
0,BANPU
1,CK
2,CKP
3,COM7
4,CPF
5,DIF
6,IVL
7,JASIF
8,JMART
9,JMT


In [9]:
sql = """
SELECT a.name,fm_date,to_date,fm_price,to_price,
a.qty,a.max_price,a.min_price,t.status,t.market
FROM sales a 
JOIN stocks t ON a.name = t.name 
WHERE to_date = '%s' AND t.status IN ("B","I", "O", "S") 
ORDER BY t.status, a.name
"""
sql = sql % yesterday
print(sql)


SELECT a.name,fm_date,to_date,fm_price,to_price,
a.qty,a.max_price,a.min_price,t.status,t.market
FROM sales a 
JOIN stocks t ON a.name = t.name 
WHERE to_date = '2023-02-07' AND t.status IN ("B","I", "O", "S") 
ORDER BY t.status, a.name



### This statement causes program to hang when there is no data

In [10]:
df = pd.read_sql(sql, conlite)
#df.set_index(["name"], inplace=True)
df['fm_date'] = pd.to_datetime(df['fm_date'])
df['to_date'] = pd.to_datetime(df['to_date'])
df.eval('diff = to_price - fm_price',inplace=True)
df['percent'] = round(df['diff']/df['fm_price']*100,2)
df.style.format(format_dict)

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,status,market,diff,percent
0,BANPU,2023-02-07,2023-02-07,11.30,11.20,"161,516,144",12.60,11.20,B,SET50,-0.10,-0.88%
1,KCE,2023-01-31,2023-02-07,54.75,57.00,"257,159,126",58.50,50.50,B,SET100,2.25,4.11%
2,RCL,2023-02-03,2023-02-07,30.75,31.00,"6,209,457",31.00,30.50,B,SET100,0.25,0.81%
3,STA,2023-02-03,2023-02-07,22.30,23.40,"27,493,867",23.40,22.20,B,SET100,1.10,4.93%
4,DIF,2023-02-02,2023-02-07,13.60,13.60,"31,473,792",13.70,13.60,I,SET,0.00,0.00%
5,IVL,2023-02-06,2023-02-07,40.50,40.50,"28,440,836",41.25,40.25,I,SET50,0.00,0.00%
6,JASIF,2023-02-03,2023-02-07,8.25,8.25,"10,368,854",8.25,8.20,I,SET,0.00,0.00%
7,JMART,2023-02-03,2023-02-07,37.25,36.75,"16,118,136",37.75,36.75,I,SET50,-0.50,-1.34%
8,JMT,2023-02-07,2023-02-07,53.75,53.00,"14,084,684",55.25,53.00,I,SET50,-0.75,-1.40%
9,ORI,2023-02-07,2023-02-07,11.70,11.70,"8,430,180",12.30,11.70,I,SET100,0.00,0.00%


In [11]:
df.shape

(20, 12)

### IF the above count not equal number of orders, there must be something incorrect

### Create daily-sales from sales

In [12]:
df[cols].style.format(format_dict)

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status
0,BANPU,2023-02-07,2023-02-07,11.30,11.20,"161,516,144",12.60,11.20,-0.88%,B
1,KCE,2023-01-31,2023-02-07,54.75,57.00,"257,159,126",58.50,50.50,4.11%,B
2,RCL,2023-02-03,2023-02-07,30.75,31.00,"6,209,457",31.00,30.50,0.81%,B
3,STA,2023-02-03,2023-02-07,22.30,23.40,"27,493,867",23.40,22.20,4.93%,B
4,DIF,2023-02-02,2023-02-07,13.60,13.60,"31,473,792",13.70,13.60,0.00%,I
5,IVL,2023-02-06,2023-02-07,40.50,40.50,"28,440,836",41.25,40.25,0.00%,I
6,JASIF,2023-02-03,2023-02-07,8.25,8.25,"10,368,854",8.25,8.20,0.00%,I
7,JMART,2023-02-03,2023-02-07,37.25,36.75,"16,118,136",37.75,36.75,-1.34%,I
8,JMT,2023-02-07,2023-02-07,53.75,53.00,"14,084,684",55.25,53.00,-1.40%,I
9,ORI,2023-02-07,2023-02-07,11.70,11.70,"8,430,180",12.30,11.70,0.00%,I


In [13]:
file_name = "daily-sales.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
one_file = one_path + file_name
osd_file = osd_path + file_name

df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(output_file, header=True, index=False)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(data_file, header=True, index=False)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(one_file, header=True, index=False)
df[cols].sort_values(['status','percent'],ascending=[True,True]).to_csv(osd_file, header=True, index=False)

In [14]:
df[cols].shape

(20, 10)

In [15]:
df.shape

(20, 12)

In [16]:
sales = df[cols]
sales

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status
0,BANPU,2023-02-07,2023-02-07,11.30,11.20,161516144,12.60,11.20,-0.88,B
1,KCE,2023-01-31,2023-02-07,54.75,57.00,257159126,58.50,50.50,4.11,B
2,RCL,2023-02-03,2023-02-07,30.75,31.00,6209457,31.00,30.50,0.81,B
3,STA,2023-02-03,2023-02-07,22.30,23.40,27493867,23.40,22.20,4.93,B
4,DIF,2023-02-02,2023-02-07,13.60,13.60,31473792,13.70,13.60,0.00,I
5,IVL,2023-02-06,2023-02-07,40.50,40.50,28440836,41.25,40.25,0.00,I
6,JASIF,2023-02-03,2023-02-07,8.25,8.25,10368854,8.25,8.20,0.00,I
7,JMART,2023-02-03,2023-02-07,37.25,36.75,16118136,37.75,36.75,-1.34,I
8,JMT,2023-02-07,2023-02-07,53.75,53.00,14084684,55.25,53.00,-1.40,I
9,ORI,2023-02-07,2023-02-07,11.70,11.70,8430180,12.30,11.70,0.00,I


In [17]:
file_name = "Price-Today.csv"
input_file = data_path + file_name
prices = pd.read_csv(input_file)
prices

,name,date,price,change,%change,open,high,low,volume,daily_volume
0,ACE,2023-02-08,2.56,0.00,0.000000,2.54,2.56,2.54,9539330,24300.34750
1,ADVANC,2023-02-08,196.00,-0.50,-0.254453,196.00,196.50,195.00,1645166,321944.39400
2,AH,2023-02-08,32.25,-0.50,-1.526718,33.00,33.00,32.25,575537,18714.97925
3,AIE,2023-02-08,3.04,0.04,1.333333,3.02,3.10,2.94,12110539,36857.44440
4,AIMIRT,2023-02-08,12.30,0.00,0.000000,12.30,12.30,12.30,257501,3167.26230
...,...,...,...,...,...,...,...,...,...,...
189,WHAIR,2023-02-08,8.00,0.00,0.000000,8.00,8.00,8.00,58100,464.80000
190,WHART,2023-02-08,11.60,0.00,0.000000,11.60,11.70,11.60,371800,4313.38000
191,WHAUP,2023-02-08,4.10,-0.02,-0.485437,4.10,4.12,4.10,526622,2164.23860
192,WICE,2023-02-08,11.60,-0.10,-0.854701,11.80,11.80,11.60,1976161,23060.68680


In [18]:
df_merge = pd.merge(sales,prices,on='name', how='inner')

In [19]:
df_merge

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status,date,price,change,%change,open,high,low,volume,daily_volume
0,BANPU,2023-02-07,2023-02-07,11.30,11.20,161516144,12.60,11.20,-0.88,B,2023-02-08,11.20,0.00,0.000000,11.30,11.40,11.10,47601625,5.336387e+05
1,KCE,2023-01-31,2023-02-07,54.75,57.00,257159126,58.50,50.50,4.11,B,2023-02-08,52.75,-4.25,-7.456140,54.50,55.25,52.50,50784876,2.737275e+06
2,RCL,2023-02-03,2023-02-07,30.75,31.00,6209457,31.00,30.50,0.81,B,2023-02-08,30.75,-0.25,-0.806452,31.00,31.25,30.75,1049670,3.244158e+04
3,STA,2023-02-03,2023-02-07,22.30,23.40,27493867,23.40,22.20,4.93,B,2023-02-08,24.00,0.60,2.564103,23.40,24.10,23.20,8936406,2.123360e+05
4,DIF,2023-02-02,2023-02-07,13.60,13.60,31473792,13.70,13.60,0.00,I,2023-02-08,13.70,0.10,0.735294,13.60,13.70,13.60,4288912,5.849629e+04
5,IVL,2023-02-06,2023-02-07,40.50,40.50,28440836,41.25,40.25,0.00,I,2023-02-08,41.00,0.50,1.234568,40.50,41.50,40.50,9271372,3.813779e+05
6,JASIF,2023-02-03,2023-02-07,8.25,8.25,10368854,8.25,8.20,0.00,I,2023-02-08,8.25,0.00,0.000000,8.20,8.25,8.20,737713,6.056239e+03
7,JMART,2023-02-03,2023-02-07,37.25,36.75,16118136,37.75,36.75,-1.34,I,2023-02-08,36.25,-0.50,-1.360544,37.00,37.00,36.00,3395145,1.237154e+05
8,JMT,2023-02-07,2023-02-07,53.75,53.00,14084684,55.25,53.00,-1.40,I,2023-02-08,52.00,-1.00,-1.886792,53.00,53.25,52.00,4507194,2.359867e+05
9,ORI,2023-02-07,2023-02-07,11.70,11.70,8430180,12.30,11.70,0.00,I,2023-02-08,11.80,0.10,0.854701,11.80,11.90,11.70,1349460,1.591319e+04


In [20]:
df_merge.columns

Index(['name', 'fm_date', 'to_date', 'fm_price', 'to_price', 'qty',
       'max_price', 'min_price', 'percent', 'status', 'date', 'price',
       'change', '%change', 'open', 'high', 'low', 'volume', 'daily_volume'],
      dtype='object')

In [23]:
colu = 'name fm_date to_date fm_price to_price qty max_price min_price percent status \
price change volume date'.split()
colu

['name',
 'fm_date',
 'to_date',
 'fm_price',
 'to_price',
 'qty',
 'max_price',
 'min_price',
 'percent',
 'status',
 'price',
 'change',
 'volume',
 'date']

In [24]:
df_merge[colu]

,name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price,percent,status,price,change,volume,date
0,BANPU,2023-02-07,2023-02-07,11.30,11.20,161516144,12.60,11.20,-0.88,B,11.20,0.00,47601625,2023-02-08
1,KCE,2023-01-31,2023-02-07,54.75,57.00,257159126,58.50,50.50,4.11,B,52.75,-4.25,50784876,2023-02-08
2,RCL,2023-02-03,2023-02-07,30.75,31.00,6209457,31.00,30.50,0.81,B,30.75,-0.25,1049670,2023-02-08
3,STA,2023-02-03,2023-02-07,22.30,23.40,27493867,23.40,22.20,4.93,B,24.00,0.60,8936406,2023-02-08
4,DIF,2023-02-02,2023-02-07,13.60,13.60,31473792,13.70,13.60,0.00,I,13.70,0.10,4288912,2023-02-08
5,IVL,2023-02-06,2023-02-07,40.50,40.50,28440836,41.25,40.25,0.00,I,41.00,0.50,9271372,2023-02-08
6,JASIF,2023-02-03,2023-02-07,8.25,8.25,10368854,8.25,8.20,0.00,I,8.25,0.00,737713,2023-02-08
7,JMART,2023-02-03,2023-02-07,37.25,36.75,16118136,37.75,36.75,-1.34,I,36.25,-0.50,3395145,2023-02-08
8,JMT,2023-02-07,2023-02-07,53.75,53.00,14084684,55.25,53.00,-1.40,I,52.00,-1.00,4507194,2023-02-08
9,ORI,2023-02-07,2023-02-07,11.70,11.70,8430180,12.30,11.70,0.00,I,11.80,0.10,1349460,2023-02-08


In [25]:
df_merge[colu].shape

(20, 14)

In [26]:
df_merge.shape

(20, 19)

In [27]:
file_name = "daily-sales-prices.csv"
data_file = data_path + file_name
output_file = csv_path + file_name
one_file = one_path + file_name
osd_file = osd_path + file_name

df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(output_file, header=True, index=False)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(data_file, header=True, index=False)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(one_file, header=True, index=False)
df_merge[colu].sort_values(['name'],ascending=[True]).to_csv(osd_file, header=True, index=False)

In [28]:
sql = """
SELECT * FROM sales"""
df_tmp = pd.read_sql(sql, conlite)
df_tmp.columns


Index(['id', 'name', 'fm_date', 'to_date', 'days', 'fm_price', 'to_price',
       'diff', 'pct', 'ttl_spread', 'avg_spread', 'max_price', 'min_price',
       'qty', 'created_at', 'updated_at', 'latest_date_id'],
      dtype='object')

### Call ruby ruby\daily-out-new.rb

In [29]:
%pwd

'C:\\Users\\User\\OneDrive\\A7\\Daily'

In [32]:
os.chdir("C:\\users\\user\\onedrive\\a7")
%pwd

'C:\\users\\user\\onedrive\\a7'

In [33]:
!ruby ruby\\daily-out-new.rb

Name      From Date    To Date   From     To     Pct     Shares    Max    Min S Action
name,fm_date,to_date,fm_price,to_price,qty,max_price,min_price


ruby//daily-out-new.rb:205:in `printf': invalid value for Float(): "fm_price" (ArgumentError)
	from ruby//daily-out-new.rb:205:in `block in <main>'
	from ruby//daily-out-new.rb:46:in `each'
	from ruby//daily-out-new.rb:46:in `<main>'
